In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import torch.optim as optim
import os


C:\Users\Amrita\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from torch.utils.data import Dataset, DataLoader

In [3]:
import torchmetrics

In [4]:
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune import CLIReporter

2025-06-16 21:53:01,971	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.6.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-06-16 21:53:02,690	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.6.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [1699]:
df=pd.read_csv('online_gaming_behavior_dataset.csv')

In [1700]:
df1=df.copy()
df1.drop('PlayerID',axis=1,inplace=True)

In [1709]:
df1['EngagementLevel'].value_counts(normalize=True)

2    0.483939
0    0.258181
1    0.257881
Name: EngagementLevel, dtype: float64

In [1702]:
df1=df1.join(pd.get_dummies(df1[['Gender','Location','GameGenre']],drop_first=True))


df1.drop(['Gender','Location','GameGenre'],axis=1,inplace=True)

In [1707]:
label_encoder = LabelEncoder()

# Fit and transform the 'color' column
df1['EngagementLevel'] = label_encoder.fit_transform(df['EngagementLevel'])

In [1710]:
df1.columns

Index(['Age', 'PlayTimeHours', 'InGamePurchases', 'GameDifficulty',
       'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel',
       'AchievementsUnlocked', 'Gender_Male', 'Location_Europe',
       'Location_Other', 'Location_USA', 'GameGenre_RPG',
       'GameGenre_Simulation', 'GameGenre_Sports', 'GameGenre_Strategy',
       'EngagementLevel'],
      dtype='object')

In [1711]:
df1.columns=['Age', 'PlayTimeHours', 'InGamePurchases', 'GameDifficulty',
       'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel',
       'AchievementsUnlocked','Gender_Male',
       'Location_Europe', 'Location_Other', 'Location_USA', 'GameGenre_RPG',
       'GameGenre_Simulation', 'GameGenre_Sports', 'GameGenre_Strategy','EngagementLevel' ]

In [1712]:
X=np.array(df1.drop('EngagementLevel',axis=1))
y=np.array(df1['EngagementLevel'])

In [1713]:
X.shape, y.shape

((40034, 16), (40034,))

In [1714]:
from torch import nn

device="cuda" if torch.cuda.is_available() else "cpu"

device

'cpu'

In [1715]:
def accuracy_fn(y_true,y_pred):
    correct=torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc=(correct/len(y_pred))*100
    return acc

In [1716]:
precision = torchmetrics.Precision(task="multiclass", average='macro', num_classes=3)
recall = torchmetrics.Recall(task="multiclass",average='macro',num_classes=3)
f1_score = torchmetrics.F1Score(task="multiclass",average='macro',num_classes=3)
con_matrix=torchmetrics.ConfusionMatrix(task="multiclass",num_classes=3)

In [1738]:
class NeuralNetwork(nn.Module):
    def __init__(self,hidden_size):
        super().__init__()
#         hsize = trial.suggest_int(name="hidden_size", low=128, high=512, step=128)
        self.linear1=nn.Linear(16,hidden_size)
        self.linear2=nn.Linear(hidden_size,hidden_size)
        self.linear3=nn.Linear(hidden_size,3)
        self.relu=nn.Sigmoid()
        
    def forward(self,x):
        x=self.linear1(x)
        x=self.relu(x)
        x=self.linear2(x)
        x=self.relu(x)
        logits=self.linear3(x)
        return logits
    

In [1739]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2, random_state=42,shuffle=True,)  ## Make the random split reproducible

len(X_train),len(X_test), len(y_train),len(y_test)

(32027, 8007, 32027, 8007)

In [1740]:
X_train=pd.DataFrame(X_train,columns=df1.columns[:-1])
y_train=pd.DataFrame(y_train,columns=[df1.columns[-1]])

X_test=pd.DataFrame(X_test,columns=df1.columns[:-1])
y_test=pd.DataFrame(y_test,columns=[df1.columns[-1]])

df_train=X_train.join(y_train)


df_test=X_test.join(y_test)

In [1741]:

class DataFrameDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
#         self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx, :-1].values # Features
        label = self.dataframe.iloc[idx, -1] # Label
        sample = torch.tensor(sample, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        return sample, label

dataset_train = DataFrameDataset(df_train)
dataset_test = DataFrameDataset(df_test)

# Create Dataset and DataLoader
train_dataloader = DataLoader(dataset_train, batch_size=20, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=20, shuffle=True)

# # Iterate through the DataLoader
# for batch_idx, (batch_samples, batch_labels) in enumerate(train_dataloader):
#     print(f"Batch {batch_idx}:")
#     print("Features:", batch_samples)
#     print("Labels:", batch_labels)

In [1792]:
def train_data(config, checkpoint_dir=None):
    net=NeuralNetwork(config['hidden_size'])
    device='cpu'
    net.to(device)
#     checkpoint_dir="C:/Users/Amrita/Downloads/Gen AI/Deep Learning Classification/results"
#     if checkpoint_dir:
#         checkpoint = torch.load(os.path.join(checkpoint_dir, "checkpoint"))
#         net.load_state_dict(checkpoint["net"])
#         optimizer.load_state_dict(checkpoint["optimizer"])
    
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.SGD(net.parameters(),lr=config['lr'])
    
    
    for epoch in range(10):  # loop over the dataset multiple times
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_dataloader, 0):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 2000 == 1999:  # print every 2000 mini-batches
                print(f"[{epoch + 1}, {i + 1}] loss: {running_loss / 2000:.3f}")
                running_loss = 0.0

        # Validation loss
        val_loss = 0.0
        val_steps = 0
        correct=0
        total=0
        for i, (inputs, labels) in enumerate(test_dataloader):
            with torch.no_grad():
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = net(inputs)
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                loss = criterion(outputs, labels.long())
                val_loss += loss.cpu().numpy()
                val_steps += 1
        tune.report({"loss": (val_loss / val_steps),"accuracy":(correct/total)},#checkpoint=checkpoint
                   )
        
#         with tune.checkpoint_dir(epoch) as checkpoint_dir:
#                 path = os.path.join(checkpoint_dir, "checkpoint")
#                 torch.save({
#                     "net": net.state_dict(),
#                     "optimizer": optimizer.state_dict()
#                 }, path)
        
    print("Finished Training")    
    

In [1793]:
config = {
    "hidden_size": tune.choice([i for i in range(128, 512,128)]), 
    "lr": tune.loguniform(1e-3, 9e-1)
}

In [ ]:
scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
#     max_t=10,
    grace_period=1,
    reduction_factor=2
)

reporter = CLIReporter(
        # parameter_columns=["l1", "l2", "lr", "batch_size"],
        metric_columns=["loss", "accuracy", "training_iteration"])


result = tune.run(
    train_data,
    storage_path='C:/Users/Amrita/Downloads/Gen AI/Deep Learning Classification/results', 
    name='first_ray', resume=True,
    resources_per_trial={"cpu":3},
    config=config,
    num_samples=10,
    scheduler=scheduler,progress_reporter=reporter,
verbose=3)

print("Best config: ", result.get_best_config(metric="loss", mode="min")) 

2025-06-16 05:00:17,019	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2025-06-16 05:00:17,135	ERROR tune_controller.py:234 -- Tried to resume experiment from directory 'C:/Users/Amrita/Downloads/Gen AI/Deep Learning Classification/results/first_ray', but no experiment state file of the form 'experiment_state-{}.json' was found. This is expected if you are launching a new experiment.
2025-06-16 05:00:17,135	ERROR tune_controller.py:235 -- Failed to restore the run state.
Traceback (most recent call last):
  File "C:\Users\Amrita\anaconda3\lib\site-packages\ray\tune\execution\tune_controller.py", line 230, in __init__
    self.resume(resume_config=resume_config)
  File "C:\Users\Amrita\anaconda3\lib\site-packages\ray\tune\execution\tune_controller.py", line 437, in resume
    raise ValueError(
ValueError: Tried t

== Status ==
Current time: 2025-06-16 05:00:18 (running for 00:00:01.62)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 05:00:48 (running for 00:00:31.83)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 05:01:19 (running for 00:01:02.08)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 05:01:49 (running for 00:01:32.47)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

...

== Status ==
Current time: 2025-06-16 11:11:06 (running for 06:10:49.28)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 11:11:36 (running for 06:11:19.77)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 11:12:07 (running for 06:11:50.27)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 11:12:37 (running for 06:12:20.48)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

== Status ==
Current time: 2025-06-16 11:13:07 (running for 06:12:50.84)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 32.000: None | Iter 16.000: None | Iter 8.000: None | Iter 4.000: None | Iter 2.000: None | Iter 1.000: None
Logical resource usage: 0/4 CPUs, 0/0 GPUs
Result logdir: C:/Users/Amrita/AppData/Local/Temp/ray/session_2025-06-16_02-18-43_297667_9256/artifacts/2025-06-16_05-00-17/first_ray/driver_artifacts
Number of trials: 10/10 (10 PENDING)
+------------------------+----------+-------+---------------+------------+
| Trial name             | status   | loc   |   hidden_size |         lr |
|------------------------+----------+-------+---------------+------------|
| train_data_b1e3f_00000 | PENDING  |       |           256 | 0.00782944 |
| train_data_b1e3f_00001 | PENDING  |       |           128 | 0.00183805 |
| train_data_b1e3f_00002 | PENDING  |       |           256 | 0.492136   |
| train_data_b1e3f_00003 | PENDING  |       |           256 | 0.064

In [1788]:
best_trial = result.get_best_trial("loss", "min", "last")
print("Best trial config: {}".format(best_trial.config))
print("Best trial final validation loss: {}".format(
    best_trial.last_result["loss"]))
print("Best trial final validation accuracy: {}".format(
    best_trial.last_result["accuracy"]))



Best trial config: {'hidden_size': 384, 'lr': 0.03881691690250773}
Best trial final validation loss: 0.3604641620097612


KeyError: 'accuracy'

In [ ]:
best_trained_model = NeuralNetwork(best_trial.config["hidden_size"], best_trial.config["l2"])

best_trained_model.to(device)

In [20]:
data.keys()

dict_keys(['trial_data', 'runner_data', 'stats'])